# Tversky Neural Networks: Implementation and Experimental Analysis

This notebook implements and evaluates Tversky Neural Networks, a novel architecture
based on Tversky's contrast model for similarity computation.

Reference: "Tversky Neural Networks" (arXiv:2506.11035v1)

### Jupyter Notebook Settings

In [1]:
from IPython.core.display import display, HTML                                    
display(HTML("<style>.container { width:100% !important; }</style>"))  
import IPython.display as display

/var/folders/ry/454yhlzx6hd15j7rjv4th0lw0000gn/T/ipykernel_26547/2003764878.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


### =============================================================================
### Libraries
### =============================================================================

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import gc
from typing import Tuple, Optional, List, Dict
import warnings

### =============================================================================
### Device Management
### =============================================================================

In [ ]:
# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

def clean_memory():
    """Clean GPU cache and run garbage collection."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

### =============================================================================
### Theory and Background
### =============================================================================

#### Tversky Similarity and Neural Networks

Tversky's contrast model measures similarity between objects based on:
- Common features: f(A ∩ B)
- Distinctive features of A: f(A - B)  
- Distinctive features of B: f(B - A)

The similarity formula is:
S(A,B) = θf(A∩B) - αf(A-B) - βf(B-A)

Key advantages:
1. **Asymmetric**: S(A,B) ≠ S(B,A) in general
2. **Feature-based**: Uses explicit feature representations
3. **Learnable**: θ, α, β parameters can be optimized

This notebook implements the Tversky Projection Layer and evaluates it on:
1. XOR problem (non-linearly separable data)
2. Biology Q&A dataset (real-world language modeling)

### =============================================================================
### Core Implementation
### =============================================================================

In [ ]:
class TverskyProjectionLayer(nn.Module):
    """
    Tversky Projection Layer implementation.
    
    This layer computes similarity between input vectors and learned prototypes
    using the differentiable Tversky similarity function.
    
    Args:
        in_features: Input dimension
        out_features: Number of prototypes (output dimension)
        num_features: Size of the learnable feature bank
        intersection_reduction: How to aggregate common features ['product', 'min', 'max', 'mean']
        difference_reduction: How to measure distinctive features ['ignorematch', 'subtractmatch']
    """
    
    def __init__(
        self, 
        in_features: int, 
        out_features: int, 
        num_features: int,
        intersection_reduction: str = 'product',
        difference_reduction: str = 'ignorematch'
    ):
        super().__init__()
        
        # Validate parameters
        valid_intersections = ['product', 'min', 'max', 'mean']
        valid_differences = ['ignorematch', 'subtractmatch']
        
        if intersection_reduction not in valid_intersections:
            raise ValueError(f"intersection_reduction must be one of {valid_intersections}")
        if difference_reduction not in valid_differences:
            raise ValueError(f"difference_reduction must be one of {valid_differences}")
        
        self.in_features = in_features
        self.out_features = out_features
        self.num_features = num_features
        self.intersection_reduction = intersection_reduction
        self.difference_reduction = difference_reduction
        
        # Learnable parameters
        self.prototypes = nn.Parameter(torch.randn(out_features, in_features))
        self.features = nn.Parameter(torch.randn(num_features, in_features))
        self.theta = nn.Parameter(torch.randn(1))  # Common features weight
        self.alpha = nn.Parameter(torch.randn(1))  # Input distinctive weight
        self.beta = nn.Parameter(torch.randn(1))   # Prototype distinctive weight
        
        # Initialize parameters
        nn.init.xavier_uniform_(self.prototypes)
        nn.init.xavier_uniform_(self.features)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass computing Tversky similarities.
        
        Args:
            x: Input tensor of shape (batch_size, in_features) or 
               (batch_size, seq_len, in_features)
        
        Returns:
            Similarity scores of shape (batch_size, out_features) or
            (batch_size, seq_len, out_features)
        """
        original_shape = x.shape
        
        # Handle 3D inputs (batch, seq, features)
        if x.dim() == 3:
            x = x.reshape(-1, original_shape[-1])
        
        batch_size = x.size(0)
        
        # Expand dimensions for broadcasting
        x_exp = x.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, D)
        prototypes_exp = self.prototypes.unsqueeze(0).unsqueeze(2)  # (1, P, 1, D)
        features_exp = self.features.unsqueeze(0).unsqueeze(0)  # (1, 1, F, D)
        
        # Calculate feature presence through dot products
        a_dot_f = (x_exp * features_exp).sum(-1)  # (B, P, F)
        b_dot_f = (prototypes_exp * features_exp).sum(-1)  # (B, P, F)
        
        # Create feature presence masks
        a_has_f = a_dot_f > 0
        b_has_f = b_dot_f > 0
        
        # Compute common features: f(A ∩ B)
        common_mask = a_has_f & b_has_f
        intersection_measures = self._compute_intersection(a_dot_f, b_dot_f)
        f_A_intersect_B = (intersection_measures * common_mask).sum(-1)
        
        # Compute distinctive features
        f_A_minus_B, f_B_minus_A = self._compute_differences(
            a_dot_f, b_dot_f, a_has_f, b_has_f, common_mask
        )
        
        # Final Tversky similarity: θf(A∩B) - αf(A-B) - βf(B-A)
        similarity = (
            self.theta * f_A_intersect_B - 
            self.alpha * f_A_minus_B - 
            self.beta * f_B_minus_A
        )
        
        # Restore original shape for 3D inputs
        if len(original_shape) == 3:
            similarity = similarity.reshape(*original_shape[:-1], self.out_features)
            
        return similarity
    
    def _compute_intersection(self, a_dot_f: torch.Tensor, b_dot_f: torch.Tensor) -> torch.Tensor:
        """Compute intersection measures based on reduction method."""
        if self.intersection_reduction == 'product':
            return a_dot_f * b_dot_f
        elif self.intersection_reduction == 'min':
            return torch.min(a_dot_f, b_dot_f)
        elif self.intersection_reduction == 'max':
            return torch.max(a_dot_f, b_dot_f)
        elif self.intersection_reduction == 'mean':
            return (a_dot_f + b_dot_f) / 2
    
    def _compute_differences(
        self, 
        a_dot_f: torch.Tensor, 
        b_dot_f: torch.Tensor,
        a_has_f: torch.Tensor, 
        b_has_f: torch.Tensor, 
        common_mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Compute distinctive feature measures."""
        if self.difference_reduction == 'ignorematch':
            # Features unique to each set
            f_A_minus_B = (a_dot_f * (a_has_f & ~b_has_f)).sum(-1)
            f_B_minus_A = (b_dot_f * (b_has_f & ~a_has_f)).sum(-1)
        else:  # subtractmatch
            # Difference in strength for shared features
            distinctive_A_mask = common_mask & (a_dot_f > b_dot_f)
            distinctive_B_mask = common_mask & (b_dot_f > a_dot_f)
            f_A_minus_B = ((a_dot_f - b_dot_f) * distinctive_A_mask).sum(-1)
            f_B_minus_A = ((b_dot_f - a_dot_f) * distinctive_B_mask).sum(-1)
        
        return f_A_minus_B, f_B_minus_A

### =============================================================================
### Experiment 1: XOR Problem with Feature Count Analysis
### =============================================================================

In [ ]:
class TverskyXORNet(nn.Module):
    """Simple network using a single Tversky layer for XOR classification."""
    
    def __init__(self, num_features: int):
        super().__init__()
        self.tversky_layer = TverskyProjectionLayer(
            in_features=2, 
            out_features=2, 
            num_features=num_features
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.tversky_layer(x)


def run_xor_experiment(feature_counts: List[int] = [1, 2, 4, 8, 16, 32, 64]) -> List[Dict]:
    """
    Test Tversky layer performance on XOR problem with different feature counts.
    
    The XOR problem is non-linearly separable, making it a good test for the
    expressive power of the Tversky similarity function.
    """
    print("=" * 60)
    print("EXPERIMENT 1: XOR CLASSIFICATION")
    print("=" * 60)
    print("Testing how the number of features affects Tversky layer performance on XOR.")
    
    # XOR dataset
    X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32, device=DEVICE)
    y = torch.tensor([0, 1, 1, 0], device=DEVICE)
    
    results = []
    
    for num_features in feature_counts:
        print(f"\nTesting with {num_features} features...")
        
        # Create and train model
        model = TverskyXORNet(num_features).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=0.1)
        criterion = nn.CrossEntropyLoss()
        
        # Training loop
        for epoch in range(1000):
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
        
        # Evaluate
        with torch.no_grad():
            outputs = model(X)
            predictions = torch.argmax(outputs, dim=1)
            accuracy = (predictions == y).float().mean().item()
        
        results.append({
            'num_features': num_features,
            'accuracy': accuracy,
            'final_loss': loss.item()
        })
        
        print(f"  Final accuracy: {accuracy:.1%}")
        clean_memory()
    
    return results


def plot_xor_results(results: List[Dict]):
    """Visualize XOR experiment results."""
    feature_counts = [r['num_features'] for r in results]
    accuracies = [r['accuracy'] for r in results]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy plot
    ax1.plot(feature_counts, accuracies, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Features (|Ω|)')
    ax1.set_ylabel('Final Accuracy')
    ax1.set_title('XOR Classification: Accuracy vs Feature Count')
    ax1.set_xscale('log', base=2)
    ax1.set_xticks(feature_counts)
    ax1.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax1.axhline(y=1.0, color='r', linestyle='--', alpha=0.7, label='Perfect Score')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Summary table
    ax2.axis('tight')
    ax2.axis('off')
    table_data = [[f"{r['num_features']}", f"{r['accuracy']:.1%}"] for r in results]
    table = ax2.table(cellText=table_data, colLabels=['Features', 'Accuracy'], 
                     cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    ax2.set_title('Results Summary')
    
    plt.tight_layout()
    plt.show()

### =============================================================================
### Experiment 2: Biology Q&A Dataset
### =============================================================================

In [ ]:
def load_vocabulary(cache_path: str) -> Tuple[Optional[Dict], Optional[Dict], Optional[Dict]]:
    """Load vocabulary from cache file."""
    try:
        with open(cache_path, 'r') as f:
            cache_data = json.load(f)
    except FileNotFoundError:
        print(f"ERROR: Vocabulary file not found at {cache_path}")
        return None, None, None
    
    vocab_data = {
        'base_chars': cache_data['base_chars'],
        'ngrams': cache_data['ngrams'],
        'vocab_size': cache_data['vocab_size'],
        'special_tokens': cache_data['special_tokens'],
        'final_vocab_list': cache_data['final_vocab_list']
    }
    
    # Create mappings
    char_to_id = {tok: i for i, tok in enumerate(cache_data['final_vocab_list'], start=1)}
    char_to_id['<PAD>'] = 0
    for name, token_id in cache_data['special_tokens'].items():
        char_to_id[f'<{name}>'] = token_id
    
    id_to_char = {v: k for k, v in char_to_id.items()}
    
    return vocab_data, char_to_id, id_to_char


def simple_tokenize(text: str, char_to_id: Dict, ngrams: List[str]) -> List[int]:
    """Simple tokenizer that prioritizes n-grams over individual characters."""
    tokens = []
    i = 0
    sorted_ngrams = sorted(ngrams, key=len, reverse=True)
    
    while i < len(text):
        matched = False
        for ngram in sorted_ngrams:
            if text[i:i+len(ngram)] == ngram and ngram in char_to_id:
                tokens.append(char_to_id[ngram])
                i += len(ngram)
                matched = True
                break
        
        if not matched:
            char = text[i]
            if char in char_to_id:
                tokens.append(char_to_id[char])
            i += 1
    
    return tokens


def create_qa_dataset(
    csv_path: str, 
    char_to_id: Dict, 
    ngrams: List[str], 
    seq_len: int = 64
) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
    """Create tokenized sequences from biology Q&A data."""
    try:
        df = pd.read_csv(csv_path).fillna("")
    except FileNotFoundError:
        print(f"ERROR: Dataset file not found at {csv_path}")
        return None, None
    
    inputs, targets = [], []
    special_tokens = {
        'start': char_to_id['<START_TOKEN>'],
        'question': char_to_id['<QUESTION_TOKEN>'],
        'answer': char_to_id['<ANSWER_TOKEN>'],
        'end': char_to_id['<END_TOKEN>'],
        'pad': char_to_id['<PAD>']
    }
    
    for _, row in df.iterrows():
        q_tokens = simple_tokenize(row['Question'].lower(), char_to_id, ngrams)
        a_tokens = simple_tokenize(row['Answer'].lower(), char_to_id, ngrams)
        
        if not q_tokens or not a_tokens:
            continue
        
        # Create sequence: <START> <QUESTION> q_tokens <ANSWER> a_tokens <END>
        full_seq = ([special_tokens['start']] + [special_tokens['question']] + 
                   q_tokens + [special_tokens['answer']] + a_tokens + [special_tokens['end']])
        
        if len(full_seq) < 2:
            continue
        
        input_seq = full_seq[:-1]
        target_seq = full_seq[1:]
        
        # Pad or truncate to seq_len
        input_seq = (input_seq[:seq_len] + [special_tokens['pad']] * seq_len)[:seq_len]
        target_seq = (target_seq[:seq_len] + [special_tokens['pad']] * seq_len)[:seq_len]
        
        inputs.append(input_seq)
        targets.append(target_seq)
    
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)


class BiologyTransformer(nn.Module):
    """Minimal transformer for biology Q&A with Tversky output head."""
    
    def __init__(self, vocab_size: int, hidden_dim: int, num_features: int):
        super().__init__()
        self.vocab_size = vocab_size
        
        # Embedding layers
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embedding = nn.Embedding(512, hidden_dim)
        
        # Single transformer layer
        self.transformer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4,
            dim_feedforward=hidden_dim * 2,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )
        
        # Tversky output head
        self.output_head = TverskyProjectionLayer(hidden_dim, vocab_size, num_features)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.shape
        
        # Create position embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)
        
        # Embed and add positional encoding
        x = self.dropout(self.embedding(x) + self.pos_embedding(positions))
        
        # Create causal mask
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        
        # Apply transformer
        x = self.transformer(x, src_mask=causal_mask)
        
        # Output projection
        return self.output_head(x)

def run_biology_experiment(
    vocab_data: Dict, 
    char_to_id: Dict, 
    csv_path: str,
    feature_counts: List[int] = [2, 4, 8, 16, 32, 48, 64, 96],
    batch_size: int = 128
) -> List[Dict]:
    """
    Test Tversky layer on biology Q&A dataset with different feature counts.
    
    Args:
        vocab_data: Vocabulary metadata dictionary
        char_to_id: Character to ID mapping
        csv_path: Path to biology Q&A CSV file
        feature_counts: List of feature counts to test
        batch_size: Training batch size
        
    Returns:
        List of results dictionaries with metrics for each feature count
    """
    print("\n" + "=" * 60)
    print("EXPERIMENT 2: BIOLOGY Q&A LANGUAGE MODELING")
    print("=" * 60)
    print("Testing Tversky layers on real-world biology question-answering data.")
    
    # Load and split data
    print("\nLoading and preprocessing data...")
    inputs, targets = create_qa_dataset(csv_path, char_to_id, vocab_data['ngrams'])
    if inputs is None:
        return []
    
    # Train/test split
    split_idx = int(len(inputs) * 0.8)
    train_inputs, train_targets = inputs[:split_idx], targets[:split_idx]
    test_inputs, test_targets = inputs[split_idx:], targets[split_idx:]
    
    print(f"Dataset: {len(train_inputs)} train, {len(test_inputs)} test sequences")
    print(f"Vocabulary size: {vocab_data['vocab_size']}")
    
    results = []
    
    for num_features in feature_counts:
        print(f"\n--- Testing with num_features = {num_features} ---")
        
        # Create model
        model = BiologyTransformer(
            vocab_size=vocab_data['vocab_size'],
            hidden_dim=64,
            num_features=num_features
        ).to(DEVICE)
        
        # Training setup
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
        criterion = nn.CrossEntropyLoss(ignore_index=char_to_id['<PAD>'])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)
        
        # Training loop with batching
        epochs = 25
        model.train()
        for epoch in tqdm(range(epochs), desc="Training"):
            # Shuffle data
            perm = torch.randperm(train_inputs.size(0))
            shuffled_inputs = train_inputs[perm]
            shuffled_targets = train_targets[perm]
            
            for i in range(0, len(shuffled_inputs), batch_size):
                batch_x = shuffled_inputs[i:i+batch_size].to(DEVICE)
                batch_y = shuffled_targets[i:i+batch_size].to(DEVICE)
                
                optimizer.zero_grad()
                logits = model(batch_x)
                loss = criterion(logits.view(-1, vocab_data['vocab_size']), batch_y.view(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            
            scheduler.step()
        
        # Evaluation with batching
        model.eval()
        total_loss, total_correct, total_tokens = 0, 0, 0
        
        with torch.no_grad():
            for i in range(0, len(test_inputs), batch_size):
                batch_x = test_inputs[i:i+batch_size].to(DEVICE)
                batch_y = test_targets[i:i+batch_size].to(DEVICE)
                
                logits = model(batch_x)
                loss = criterion(logits.view(-1, vocab_data['vocab_size']), batch_y.view(-1))
                total_loss += loss.item() * batch_x.size(0)
                
                predictions = torch.argmax(logits, dim=-1)
                mask = batch_y != char_to_id['<PAD>']
                total_correct += ((predictions == batch_y) & mask).sum().item()
                total_tokens += mask.sum().item()
        
        accuracy = total_correct / total_tokens if total_tokens > 0 else 0
        avg_loss = total_loss / len(test_inputs)
        
        results.append({
            'num_features': num_features,
            'accuracy': accuracy,
            'test_loss': avg_loss
        })
        
        print(f"  -> Test Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2%}")
        clean_memory()
    
    return results


def plot_biology_results(results: List[Dict]):
    """
    Visualize biology experiment results with dual plots.
    
    Args:
        results: List of result dictionaries from run_biology_experiment
    """
    if not results:
        print("No results to plot.")
        return
    
    feature_counts = [r['num_features'] for r in results]
    accuracies = [r['accuracy'] for r in results]
    losses = [r['test_loss'] for r in results]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Accuracy plot
    ax1.plot(feature_counts, accuracies, 'ro-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Features (|Ω|)')
    ax1.set_ylabel('Test Accuracy')
    ax1.set_title('Biology Q&A: Model Performance vs Feature Count')
    ax1.grid(True, alpha=0.3)
    
    # Add value labels on points
    for i, acc in enumerate(accuracies):
        ax1.annotate(f'{acc:.1%}', (feature_counts[i], acc), 
                    textcoords="offset points", xytext=(0,10), ha='center')
    
    # Loss plot
    ax2.plot(feature_counts, losses, 'go-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Features (|Ω|)')
    ax2.set_ylabel('Test Loss')
    ax2.set_title('Biology Q&A: Test Loss vs Feature Count')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on points
    for i, loss in enumerate(losses):
        ax2.annotate(f'{loss:.2f}', (feature_counts[i], loss), 
                    textcoords="offset points", xytext=(0,10), ha='center')
    
    plt.tight_layout()
    plt.show()

### =============================================================================
### MAIN EXECUTION
### =============================================================================

In [ ]:
def main():
    """
    Main execution function that runs all experiments and displays results.
    
    This function coordinates the entire experimental pipeline:
    1. Runs XOR classification experiment
    2. Attempts to run biology Q&A experiment (if data available)
    3. Displays comprehensive results and analysis
    """
    print("Tversky Neural Networks: Implementation and Experimental Analysis")
    print("=" * 70)
    
    # Run XOR experiment
    print("\nRunning XOR classification experiment...")
    xor_results = run_xor_experiment()
    
    print("\nXOR Results Summary:")
    print("Num Features | Accuracy")
    print("-------------|----------")
    for result in xor_results:
        print(f"{result['num_features']:^12} | {result['accuracy']:^8.1%}")
    
    plot_xor_results(xor_results)
    
    # Run biology experiment (if data is available)
    vocab_cache_path = 'pipeline_v31_full/vocabulary/vocabulary_cache.json'
    csv_path = 'blt_data/biology_qa_3000_augmented.csv'
    
    print(f"\nAttempting to load vocabulary from: {vocab_cache_path}")
    vocab_data, char_to_id, id_to_char = load_vocabulary(vocab_cache_path)
    
    if vocab_data is not None:
        print("✓ Vocabulary loaded successfully")
        bio_results = run_biology_experiment(vocab_data, char_to_id, csv_path)
        
        if bio_results:
            print("\nBiology Q&A Results Summary:")
            print("Num Features | Test Accuracy | Test Loss")
            print("-------------|---------------|----------")
            for result in bio_results:
                print(f"{result['num_features']:^12} | "
                      f"{result['accuracy']:^13.2%} | "
                      f"{result['test_loss']:^9.4f}")
            
            plot_biology_results(bio_results)
            
            # Find optimal feature count
            best_result = max(bio_results, key=lambda x: x['accuracy'])
            print(f"\nOptimal configuration: {best_result['num_features']} features")
            print(f"Best accuracy: {best_result['accuracy']:.2%}")
            
            # Performance trend analysis
            if len(bio_results) > 1:
                print(f"\nPerformance trend:")
                for i in range(1, len(bio_results)):
                    curr_acc = bio_results[i]['accuracy']
                    prev_acc = bio_results[i-1]['accuracy']
                    change = curr_acc - prev_acc
                    direction = "up" if change > 0 else "down" if change < 0 else "flat"
                    print(f"   {bio_results[i-1]['num_features']:2d} -> "
                          f"{bio_results[i]['num_features']:2d} features: "
                          f"{direction} {change:+.1%}")
        else:
            print("No biology results generated. Check data loading")
    else:
        print("Skipping biology experiment due to missing data files.")